In [22]:
import numpy as np
import pandas as pd
import glob
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

In [23]:
# папка з датасетами
DATA_PATH = "datasets/*.csv"   # змінити якщо інший шлях

files = sorted(glob.glob(DATA_PATH))

dfs = []
for f in files:
    df_temp = pd.read_csv(f, low_memory=False)
    df_temp["SeasonFile"] = f
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Loaded files:", len(files))
print("Total matches:", len(df))

Loaded files: 21
Total matches: 7761


/var/folders/st/shh7qfws6gs0x69q2x2sp69w0000gn/T/ipykernel_20341/1942832908.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp["SeasonFile"] = f


In [24]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df = df.dropna(subset=["Date","HomeTeam","AwayTeam","FTR"])

df = df.sort_values("Date").reset_index(drop=True)

/var/folders/st/shh7qfws6gs0x69q2x2sp69w0000gn/T/ipykernel_20341/2222272732.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


In [25]:
LABEL_MAP = {"H":0,"D":1,"A":2}

df["y"] = df["FTR"].map(LABEL_MAP)
df = df.dropna(subset=["y"])
df["y"] = df["y"].astype(int)

print(df["y"].value_counts())

y
0    3552
2    2337
1    1871
Name: count, dtype: int64


/var/folders/st/shh7qfws6gs0x69q2x2sp69w0000gn/T/ipykernel_20341/3997914059.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["y"] = df["FTR"].map(LABEL_MAP)


In [26]:
WINDOW = 20

POST_MATCH_COLS = {
    "goals_for": ("FTHG", "FTAG"),
    "goals_against": ("FTAG", "FTHG"),
    "shots_for": ("HS", "AS"),
    "shots_against": ("AS", "HS"),
    "shotsT_for": ("HST", "AST"),
    "shotsT_against": ("AST", "HST"),
    "corners_for": ("HC", "AC"),
    "corners_against": ("AC", "HC"),
    "fouls_for": ("HF", "AF"),
    "fouls_against": ("AF", "HF"),
    "yellow_for": ("HY", "AY"),
    "yellow_against": ("AY", "HY"),
    "red_for": ("HR", "AR"),
    "red_against": ("AR", "HR"),
}

In [27]:
available = set(df.columns)
usable_keys = [k for k, (h, a) in POST_MATCH_COLS.items() if h in available and a in available]

print("Usable stat groups:", usable_keys)

Usable stat groups: ['goals_for', 'goals_against', 'shots_for', 'shots_against', 'shotsT_for', 'shotsT_against', 'corners_for', 'corners_against', 'fouls_for', 'fouls_against', 'yellow_for', 'yellow_against', 'red_for', 'red_against']


In [28]:
def home_points(ftr):
    return 3 if ftr == "H" else 1 if ftr == "D" else 0

def away_points(ftr):
    return 3 if ftr == "A" else 1 if ftr == "D" else 0

df = df.copy()
df["HomePts"] = df["FTR"].apply(home_points)
df["AwayPts"] = df["FTR"].apply(away_points)

In [29]:
df = df.sort_values("Date").reset_index(drop=True)

In [30]:
elo = {}
INITIAL_ELO = 1500
K = 20
HOME_ADVANTAGE = 100

for idx, r in df.iterrows():
    home_team = r["HomeTeam"]
    away_team = r["AwayTeam"]

    elo_home = elo.get(home_team, INITIAL_ELO)
    elo_away = elo.get(away_team, INITIAL_ELO)

    df.loc[idx, "EloHome"] = elo_home
    df.loc[idx, "EloAway"] = elo_away
    df.loc[idx, "EloDiff"] = elo_home - elo_away

    expected_home = 1 / (1 + 10 ** (((elo_away) - (elo_home + HOME_ADVANTAGE)) / 400))

    if r["FTR"] == "H":
        score_home = 1.0
    elif r["FTR"] == "D":
        score_home = 0.5
    else:
        score_home = 0.0

    elo[home_team] = elo_home + K * (score_home - expected_home)
    elo[away_team] = elo_away + K * ((1 - score_home) - (1 - expected_home))

In [31]:
rows = []

for idx, r in df.iterrows():
    rec_h = {
        "MatchIdx": idx,
        "Date": r["Date"],
        "Team": r["HomeTeam"],
        "IsHome": 1,
        "Pts": r["HomePts"],
    }

    rec_a = {
        "MatchIdx": idx,
        "Date": r["Date"],
        "Team": r["AwayTeam"],
        "IsHome": 0,
        "Pts": r["AwayPts"],
    }

    for k in usable_keys:
        home_col, away_col = POST_MATCH_COLS[k]
        rec_h[k] = pd.to_numeric(r.get(home_col), errors="coerce")
        rec_a[k] = pd.to_numeric(r.get(away_col), errors="coerce")

    rows.append(rec_h)
    rows.append(rec_a)

long_df = pd.DataFrame(rows).sort_values(["Team", "Date", "MatchIdx"]).reset_index(drop=True)

In [32]:
feature_cols = []

for col in ["Pts"] + usable_keys:
    roll_col = f"{col}_roll{WINDOW}"
    long_df[roll_col] = (
        long_df.groupby("Team")[col]
        .apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    feature_cols.append(roll_col)

long_df[feature_cols] = long_df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

In [33]:
home_feats = long_df[long_df["IsHome"] == 1][["MatchIdx"] + feature_cols].copy().add_prefix("H_")
away_feats = long_df[long_df["IsHome"] == 0][["MatchIdx"] + feature_cols].copy().add_prefix("A_")

match_feats = df.copy()
match_feats = match_feats.merge(home_feats, left_index=True, right_on="H_MatchIdx", how="left")
match_feats = match_feats.merge(away_feats, left_index=True, right_on="A_MatchIdx", how="left")
match_feats = match_feats.drop(columns=["H_MatchIdx", "A_MatchIdx"], errors="ignore")

In [34]:
X_cols = [c for c in match_feats.columns if c.startswith("H_") or c.startswith("A_")]
X_cols += ["EloHome", "EloAway", "EloDiff"]

match_feats[X_cols] = match_feats[X_cols].fillna(0.0)

X = match_feats[X_cols].to_numpy(dtype=np.float32)
y = match_feats["y"].to_numpy(dtype=np.int64)

print("X shape:", X.shape, "y shape:", y.shape)

X shape: (7760, 33) y shape: (7760,)


In [35]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

test_indices = np.arange(split_idx, len(X))

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (6208, 33) (6208,)
Test: (1552, 33) (1552,)


In [36]:
tree_models = {
    "DecisionTree_depth5": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ),
    "DecisionTree_depth10": DecisionTreeClassifier(
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ),
    "RandomForest_200": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees_200": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    ),
}

In [37]:
tree_models["XGBoost"] = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="multi:softprob",  # важливо для 3 класів
    num_class=3,

    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

In [38]:
results = []
trained_models = {}
all_model_outputs = {}

for name, model in tree_models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    acc = accuracy_score(y_test, preds)
    ll = log_loss(y_test, probs, labels=[0, 1, 2])

    results.append({
        "model": name,
        "accuracy": acc,
        "logloss": ll
    })

    all_model_outputs[name] = {
        "preds": preds,
        "probs": probs
    }

    print(f"\n{name}")
    print("Accuracy:", round(acc, 3))
    print("LogLoss:", round(ll, 3))
    print(classification_report(y_test, preds, target_names=["HomeWin", "Draw", "AwayWin"]))
    print("Confusion matrix:\n", confusion_matrix(y_test, preds))


DecisionTree_depth5
Accuracy: 0.477
LogLoss: 1.238
              precision    recall  f1-score   support

     HomeWin       0.63      0.54      0.58       704
        Draw       0.26      0.41      0.32       349
     AwayWin       0.55      0.44      0.49       499

    accuracy                           0.48      1552
   macro avg       0.48      0.46      0.46      1552
weighted avg       0.52      0.48      0.49      1552

Confusion matrix:
 [[377 233  94]
 [119 144  86]
 [103 177 219]]

DecisionTree_depth10
Accuracy: 0.399
LogLoss: 6.102
              precision    recall  f1-score   support

     HomeWin       0.58      0.36      0.44       704
        Draw       0.21      0.29      0.24       349
     AwayWin       0.42      0.54      0.47       499

    accuracy                           0.40      1552
   macro avg       0.40      0.39      0.39      1552
weighted avg       0.45      0.40      0.41      1552

Confusion matrix:
 [[252 242 210]
 [ 92 100 157]
 [ 89 142 268]]

Ra

In [39]:
results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)
print(results_df)

                  model  accuracy   logloss
0      GradientBoosting  0.539948  0.973774
1               XGBoost  0.523196  0.997934
2      RandomForest_200  0.519974  0.990939
3        ExtraTrees_200  0.506443  1.002135
4   DecisionTree_depth5  0.476804  1.238080
5  DecisionTree_depth10  0.399485  6.101533


In [40]:
test_odds = match_feats.iloc[test_indices][["B365H", "B365D", "B365A"]].copy()

mask = test_odds[["B365H", "B365D", "B365A"]].notna().all(axis=1).values

pH_raw = 1 / test_odds.loc[mask, "B365H"].values
pD_raw = 1 / test_odds.loc[mask, "B365D"].values
pA_raw = 1 / test_odds.loc[mask, "B365A"].values

s = pH_raw + pD_raw + pA_raw
pH = pH_raw / s
pD = pD_raw / s
pA = pA_raw / s

market_probs = np.vstack([pH, pD, pA]).T
market_preds = market_probs.argmax(axis=1)

market_acc = accuracy_score(y_test[mask], market_preds)
market_ll = log_loss(y_test[mask], market_probs, labels=[0, 1, 2])

favorite_no_draw = np.where(pH > pA, 0, 2)
fav_acc = accuracy_score(y_test[mask], favorite_no_draw)

print("Market (B365) accuracy:", round(market_acc, 3))
print("Market (B365) logloss:", round(market_ll, 3))
print("Favorite(no draw) accuracy:", round(fav_acc, 3))

Market (B365) accuracy: 0.568
Market (B365) logloss: 0.946
Favorite(no draw) accuracy: 0.566


In [41]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

best_preds = all_model_outputs[best_model_name]["preds"]
best_probs = all_model_outputs[best_model_name]["probs"]

best_acc = accuracy_score(y_test, best_preds)
best_ll = log_loss(y_test, best_probs, labels=[0, 1, 2])

best_acc_masked = accuracy_score(y_test[mask], best_preds[mask])
best_ll_masked = log_loss(y_test[mask], best_probs[mask], labels=[0, 1, 2])

print("Best tree model:", best_model_name)
print("Best tree accuracy:", round(best_acc, 3))
print("Best tree logloss:", round(best_ll, 3))
print("Best tree acc (odds available):", round(best_acc_masked, 3))
print("Best tree logloss (odds available):", round(best_ll_masked, 3))
print("Market (B365) accuracy:", round(market_acc, 3))
print("Market (B365) logloss:", round(market_ll, 3))
print("Favorite(no draw) accuracy:", round(fav_acc, 3))

Best tree model: GradientBoosting
Best tree accuracy: 0.54
Best tree logloss: 0.974
Best tree acc (odds available): 0.54
Best tree logloss (odds available): 0.974
Market (B365) accuracy: 0.568
Market (B365) logloss: 0.946
Favorite(no draw) accuracy: 0.566


In [42]:
best_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

best_model.fit(X_train, y_train)
preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds), 3))
print("LogLoss:", round(log_loss(y_test, probs, labels=[0,1,2]), 3))
print(classification_report(y_test, preds, target_names=["HomeWin", "Draw", "AwayWin"]))
print("Confusion matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.54
LogLoss: 0.974
              precision    recall  f1-score   support

     HomeWin       0.58      0.73      0.65       704
        Draw       0.34      0.09      0.14       349
     AwayWin       0.51      0.59      0.54       499

    accuracy                           0.54      1552
   macro avg       0.48      0.47      0.44      1552
weighted avg       0.50      0.54      0.50      1552

Confusion matrix:
 [[516  36 152]
 [186  30 133]
 [185  22 292]]


## Robust Evaluation: Walk-Forward, Calibration, Threshold Tuning

This block adds a realistic time-series evaluation and improves probability quality for 3-way outcome prediction (H/D/A).

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, classification_report
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Ensure required arrays exist even if notebook cells were run out of order
if 'X' not in globals() or 'y' not in globals():
    raise RuntimeError('X/y are missing. Run feature-engineering cells first.')

if 'split_idx' not in globals():
    split_idx = int(len(X) * 0.8)

if 'X_train' not in globals() or 'X_test' not in globals() or 'y_train' not in globals() or 'y_test' not in globals():
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

def evaluate_predictions(y_true, preds, probs, name):
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, preds),
        'balanced_acc': balanced_accuracy_score(y_true, preds),
        'f1_macro': f1_score(y_true, preds, average='macro'),
        'log_loss': log_loss(y_true, probs, labels=[0, 1, 2]),
    }

# 1) Walk-forward validation (time-series CV)
tscv = TimeSeriesSplit(n_splits=5)
wf_rows = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), start=1):
    wf_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
    wf_model.fit(X[tr_idx], y[tr_idx])
    wf_preds = wf_model.predict(X[te_idx])
    wf_probs = wf_model.predict_proba(X[te_idx])
    row = evaluate_predictions(y[te_idx], wf_preds, wf_probs, f'Fold{fold}')
    row['train_size'] = len(tr_idx)
    row['test_size'] = len(te_idx)
    wf_rows.append(row)

wf_df = pd.DataFrame(wf_rows)
print('Walk-forward folds:')
print(wf_df[['model', 'train_size', 'test_size', 'accuracy', 'f1_macro', 'log_loss']].to_string(index=False))
print('\nWalk-forward mean metrics:')
print(wf_df[['accuracy', 'balanced_acc', 'f1_macro', 'log_loss']].mean().to_string())
print('\nWalk-forward std metrics:')
print(wf_df[['accuracy', 'balanced_acc', 'f1_macro', 'log_loss']].std().to_string())

# 2) Baseline holdout model
base_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
base_model.fit(X_train, y_train)
base_probs = base_model.predict_proba(X_test)
base_preds = base_probs.argmax(axis=1)
base_metrics = evaluate_predictions(y_test, base_preds, base_probs, 'GB_baseline_holdout')

# 3) Probability calibration on holdout protocol
# Use a time split inside train set: early chunk for fit, recent chunk for calibration
cal_split = int(len(X_train) * 0.8)
X_fit, y_fit = X_train[:cal_split], y_train[:cal_split]
X_cal, y_cal = X_train[cal_split:], y_train[cal_split:]

cal_base = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
cal_base.fit(X_fit, y_fit)
cal_model = CalibratedClassifierCV(FrozenEstimator(cal_base), method='isotonic', cv=None)
cal_model.fit(X_cal, y_cal)
cal_probs = cal_model.predict_proba(X_test)
cal_preds = cal_probs.argmax(axis=1)
cal_metrics = evaluate_predictions(y_test, cal_preds, cal_probs, 'GB_calibrated_holdout')

# 4) Draw-threshold tuning (maximize macro-F1 on calibration slice)
fit_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
fit_model.fit(X_fit, y_fit)
probs_cal_slice = fit_model.predict_proba(X_cal)

best_t = 0.33
best_score = -1.0
for t in np.arange(0.20, 0.46, 0.01):
    p = probs_cal_slice.argmax(axis=1).copy()
    draw_mask = probs_cal_slice[:, 1] >= t
    p[draw_mask] = 1
    not_draw = ~draw_mask
    p[not_draw] = np.where(probs_cal_slice[not_draw, 0] >= probs_cal_slice[not_draw, 2], 0, 2)
    score = f1_score(y_cal, p, average='macro')
    if score > best_score:
        best_score = score
        best_t = t

probs_test = fit_model.predict_proba(X_test)
thr_preds = probs_test.argmax(axis=1).copy()
draw_mask_test = probs_test[:, 1] >= best_t
thr_preds[draw_mask_test] = 1
not_draw_test = ~draw_mask_test
thr_preds[not_draw_test] = np.where(probs_test[not_draw_test, 0] >= probs_test[not_draw_test, 2], 0, 2)
thr_metrics = evaluate_predictions(y_test, thr_preds, probs_test, f'GB_draw_threshold_t={best_t:.2f}')

# Final comparison table
comparison = pd.DataFrame([base_metrics, cal_metrics, thr_metrics]).sort_values('f1_macro', ascending=False).reset_index(drop=True)
print('\nHoldout comparison (sorted by f1_macro):')
print(comparison.to_string(index=False))

print('\nBest draw threshold from calibration slice:', round(best_t, 2))
print('\nClassification report (best by f1_macro):')
best_name = comparison.iloc[0]['model']
if 'threshold' in best_name:
    print(classification_report(y_test, thr_preds, target_names=['HomeWin', 'Draw', 'AwayWin']))
elif 'calibrated' in best_name:
    print(classification_report(y_test, cal_preds, target_names=['HomeWin', 'Draw', 'AwayWin']))
else:
    print(classification_report(y_test, base_preds, target_names=['HomeWin', 'Draw', 'AwayWin']))


Walk-forward folds:
model  train_size  test_size  accuracy  f1_macro  log_loss
Fold1        1295       1293  0.477958  0.406998  1.073192
Fold2        2588       1293  0.525909  0.409413  0.997162
Fold3        3881       1293  0.524362  0.417867  0.988828
Fold4        5174       1293  0.527456  0.402225  0.998393
Fold5        6467       1293  0.532096  0.421513  0.983983

Walk-forward mean metrics:
accuracy        0.517556
balanced_acc    0.446876
f1_macro        0.411603
log_loss        1.008312

Walk-forward std metrics:
accuracy        0.022324
balanced_acc    0.012379
f1_macro        0.007928
log_loss        0.036755

Holdout comparison (sorted by f1_macro):
                   model  accuracy  balanced_acc  f1_macro  log_loss
GB_draw_threshold_t=0.29  0.482603      0.453340  0.455365  0.989462
     GB_baseline_holdout  0.539948      0.468028  0.442900  0.973774
   GB_calibrated_holdout  0.535438      0.457360  0.400081  1.096585

Best draw threshold from calibration slice: 0.29

Cl